In [8]:
from itertools import product
from pydantic import BaseModel
from typing import List
import numpy as np

In [61]:
from typing import List
from pydantic import BaseModel, ValidationError
import numpy as np
from datetime import datetime, timedelta
import random
from typing import Optional
import uuid 


def compute_lognormal_parameters(mean: float, cv: float):
    """
    Calculate mu and sigma for lognormal distribution given mean and cv.
    Adapted from: https://www.johndcook.com/blog/2022/02/24/find-log-normal-parameters/

    """
    variance = (mean * cv) ** 2
    sigma2 = np.log(1 + variance / (mean**2))
    sigma = np.sqrt(sigma2)
    mu = np.log(mean) - sigma2 / 2
    return (mu.item(), sigma.item())

class UUIDGenerator:
    """
    Helper class to generate UUIDs with a specified length of the UUID string.
    The UUID is generated using the uuid4 method and then truncated to the specified length.
    Optional to add prefix and suffix to the generated UUID, which will add to the total length.
    """

    def __init__(self, id_length: int = 10):
        self.id_length = id_length

    def generate_id(
        self, prefix: Optional[str] = "", suffix: Optional[str] = ""
    ) -> str:
        id = prefix + str(uuid.uuid4().hex)[: self.id_length] + suffix
        return id

class GroupProfiles(BaseModel):
    name: List[str]
    txn_mean_low: List[float]
    txn_mean_high: List[float]
    txn_cv_low: List[float]
    txn_cv_high: List[float]
    txn_lambda: List[float]


class Customer:
    def __init__(self, profile: dict):
        self.profile = profile
        self._date_format = "%Y-%m-%d"
        self._timestamp_format = "%Y-%m-%d %H:%M:%S"
        self._big_ticket_proba = 0.005
        self._big_ticket_multiplier = 10  # Adjust the multiplier as needed
        self._uuid_generator = UUIDGenerator(id_length=10)

    def generate_txn_value(self):
        """
        Generate a single transaction value for a customer based on their profile
        In addition, there is a fixed probability (0.5%) of a big-ticket item
        """

        if (
            np.random.rand() < self._big_ticket_proba
        ):  # rand() generates from uniform [0,1), thus there is 0.5% chance that the value is less than 0.005

            # Generate a big-ticket item
            big_ticket_mean = self.profile["txn_mean"] * self._big_ticket_multiplier
            big_ticket_sigma = self.profile["txn_sigma"]
            txn_value = round(
                np.random.lognormal(
                    mean=np.log(big_ticket_mean),
                    sigma=big_ticket_sigma,
                ),
                2,
            )
        else:
            # Generate a regular transaction
            txn_value = round(
                np.random.lognormal(
                    mean=np.log(self.profile["txn_mean"]),
                    sigma=self.profile["txn_sigma"],
                ),
                2,
            )
        return txn_value

    def generate_current_txn(self):
        """
        Generate a single transaction for a customer based on their profile with the current timestamp
        """
        txn_value = self.generate_txn_value()
        txn_timestamp = datetime.now().strftime(self._timestamp_format)
        txn_date = datetime.now().strftime(self._date_format)
        return {
            "txn_id": self._uuid_generator.generate_id(prefix="t_"),
            "txn_timestamp": txn_timestamp,
            "txn_date": txn_date,
            "txn_value": txn_value,
            "txn_fraud": 0,
            "txn_fraud_scenario": 0,
        }

    def generate_batch_txns(
        self, start_date: str = "2024-01-01", num_days: int = 30
    ) -> List[dict]:
        """
        Gererate a list of customer transactions for a given number of days

        Parameters
        -----------
        start_date: str
            The starting date of the transactions in the format 'YYYY-MM-DD'
        num_days: int
            The number of days for which to generate transactions

        Returns
        --------
        List[dict]
            A list of dictionaries where each dictionary represents a transaction
        """
        batch_txns = []
        for day in range(num_days):
            num_txn = np.random.poisson(self.profile["txn_lambda"])
            if num_txn > 0:
                for _ in range(num_txn):
                    # Time of transaction: revolves around noontime, with std 20000 seconds. This is meant to simulate the fact that most transactions should occur during the day (e.g., grocery, gas, other shopping...)
                    time_txn = int(np.random.normal(86400 / 2, 20000))

                    if (time_txn > 0) and (time_txn < 86400):
                        txn_value = self.generate_txn_value()

                        txn_timestamp = datetime.fromtimestamp(
                            time_txn, tz=None
                        ) + timedelta(days=day)

                        txn_timestamp = datetime.strptime(
                            start_date, self._date_format
                        ) + timedelta(seconds=time_txn, days=day)

                        # convert to str and save
                        txn_date = txn_timestamp.strftime(self._date_format)
                        txn_timestamp = txn_timestamp.strftime(self._timestamp_format)
                        batch_txns.append(
                            {
                                "txn_id": self._uuid_generator.generate_id(prefix="t_"),
                                "txn_timestamp": txn_timestamp,
                                "txn_date": txn_date,
                                "txn_value": txn_value,
                                "txn_fraud": 0,
                                "txn_fraud_scenario": 0,
                            }
                        )
        return batch_txns


class CustomerGenerator:
    def __init__(self, spending_habits: dict):
        """
        Initialize the CustomerGenerator with a dictionary of group profiles.

        Parameters:
        -----------
        spending_habits: dict
            A dictionary containing the group profiles for generating customer transactions.
            The dictionary must have the following structure where the key names and their value types are compulsory:
            {
                'name': ['low', 'low-middle', 'middle', 'high-middle', 'high'],
                'txn_mean_low': [5, 20, 40, 60, 80],
                'txn_mean_high': [20, 40, 60, 80, 100],
                'txn_cv_low': [0.3, 0.4, 0.5, 0.6, 0.7],
                'txn_cv_high': [0.4, 0.5, 0.6, 0.7, 0.8],
                'txn_lambda': [0.25, 0.5, 1, 1.5, 2]
            }

        """
        # TODO: Change the argument name 'spending_habits' to 'spending_habit'
        # validate the spending_habits
        try:
            GroupProfiles(**spending_habits)
        except ValidationError as e:
            raise ValueError(f"Invalid spending_habits data: {e}")

        self.spending_habits = self._convert_col_to_row_oriented_profile(
            spending_habits, "name"
        )

    def generate_customer_from_profile(self, profile: dict):
        """
        Generate a customer object with a specific profile, modelled from the chosen profile name
        """

        # {'age_group': ['46-55', 0.01], 'spending_group': ['low-middle', 0.0075], 'two_fa': ['yes', 0.005]}
        spending_group = profile["spending_group"][0]
        assert spending_group in self.spending_habits.keys(), "Profile name not found"

        group_spending_habit = self.spending_habits[spending_group]
        txn_mean = round(
            np.random.uniform(group_spending_habit ["txn_mean_low"], group_spending_habit ["txn_mean_high"]), 2
        )
        cv = round(np.random.uniform(group_spending_habit ["txn_cv_low"], group_spending_habit ["txn_cv_high"]), 2)
        txn_mu, txn_sigma = compute_lognormal_parameters(txn_mean, cv)
        txn_sigma = txn_sigma
        txn_lambda = group_spending_habit["txn_lambda"]

        customer_spending_habit = {
            "txn_mean": txn_mean,
            "txn_mu": txn_mu,
            "txn_sigma": txn_sigma,
            "txn_lambda": txn_lambda,
        }
        customer_profile = {}
        # combine the customer_spending_habit with the profile into 1 dictionary
        customer_profile.update(profile)
        customer_profile.update(customer_spending_habit)
        return Customer(customer_profile)

    def _convert_col_to_row_oriented_profile(
        self, input_dict: dict, key_field: str
    ) -> dict:
        """
        Convert a column-oriented dictionary, which is more concise, to a row-oriented dictionary, which is easier to extract field-specific data from.

        Example:
        --------
        input_dict = {
            'key_field': ['A', 'B', 'C'],
            'field1': [1, 2, 3],
            'field2': [4, 5, 6]
        }
        output_dict = {
            'A': {'field1': 1, 'field2': 4},
            'B': {'field1': 2, 'field2': 5},
            'C': {'field1': 3, 'field2': 6}
        }
        """
        assert (
            key_field in input_dict
        ), f"Key field '{key_field}' not found in input dictionary"
        output_dict = {}
        key_values = input_dict[key_field]
        other_fields = {k: v for k, v in input_dict.items() if k != key_field}

        for i, key in enumerate(key_values):
            output_dict[key] = {
                field: values[i] for field, values in other_fields.items()
            }

        return output_dict


class FraudulentTxnGenerator:
    """
    Class to generate fraudulent transactions for a given customer and date based on a specific scenario.
    """

    def __init__(self):
        self._date_format = "%Y-%m-%d"
        self._timestamp_format = "%Y-%m-%d %H:%M:%S"
        self._uuid_generator = UUIDGenerator(id_length=10)

    def generate_fraudulent_txns(
        self, customer_id: str, scenario: int, date: str
    ) -> List[dict]:
        """
        Generate a batch of fraudulent transactions for a given date.
        Scenario 1: Unusual large transactions scattered through a number of days
        Scenario 2: Large transactions in quick successions with increasing amounts.
        Scenario 3: A small transaction, followed by quick successions of a large amount.

        Parameters:
        -----------
        date: str
            The date of the transactions in the format 'YYYY-MM-DD'. E.g., '2024-01-01'

        Returns:
        --------
        List[dict]
            A list of dictionaries representing the fraudulent transactions
            E.g., [{'txn_id': 't_1', 'txn_timestamp': '2024-01-01 12:00:00', 'txn_value': 100.0, 'txn_fraud': 1, 'txn_fraud_scenario': 1}]
        """

        # Generate a random timestamp within the given date
        start_time = np.random.randint(
            0, 86400
        )  # Random second in the day (0 to 86400)

        # Convert the date string to a datetime object
        date_obj = datetime.strptime(date, self._date_format)

        # Generate the fraudulent transactions
        fraudulent_txns = []
        txn_value_increment = random.choice(range(500, 2000, 500))

        if scenario == 1:
            compromised_days = np.random.randint(
                5, 14
            )  # Can be converted to user's input later
            current_date = date_obj
            # for each day within the compromised days, generate a few transactions with large values
            # the large values are arbitrarily chosen to be "nice" numbers
            for _ in range(compromised_days):
                num_txns_day = np.random.randint(1, 3)
                for i in range(num_txns_day):
                    # txn_timestamp is a random time within the day
                    txn_timestamp = current_date + timedelta(
                        seconds=np.random.randint(0, 86400)
                    )
                    txn_date = txn_timestamp.strftime(self._date_format)
                    txn_timestamp = txn_timestamp.strftime(self._timestamp_format)
                    txn_value = round(
                        random.choice(range(500, 2000, 500)), 2
                    )  # Example transaction value

                    fraudulent_txns.append(
                        {
                            "customer_id": customer_id,
                            "txn_id": self._uuid_generator.generate_id(prefix="t_"),
                            "txn_timestamp": txn_timestamp,
                            "txn_date": txn_date,
                            "txn_value": txn_value,
                            "txn_fraud": 1,
                            "txn_fraud_scenario": 1,
                        }
                    )
                current_date += timedelta(days=1)

        elif scenario == 2:
            num_txns = np.random.randint(5, 10)
            interval_seconds = np.random.randint(1, 5) * 60
            for i in range(num_txns):
                txn_timestamp = date_obj + timedelta(
                    seconds=start_time + i * np.random.randint(1, interval_seconds)
                )
                txn_date = txn_timestamp.strftime(self._date_format)
                txn_timestamp = txn_timestamp.strftime(self._timestamp_format)
                txn_value = round(
                    (i + 1) * txn_value_increment, 2
                )  # Example transaction value
                fraudulent_txns.append(
                    {
                        "customer_id": customer_id,
                        "txn_id": self._uuid_generator.generate_id(prefix="t_"),
                        "txn_timestamp": txn_timestamp,
                        "txn_date": txn_date,
                        "txn_value": txn_value,
                        "txn_fraud": 1,
                        "txn_fraud_scenario": 2,
                    }
                )
        elif scenario == 3:
            num_txns = np.random.randint(5, 10)
            interval_seconds = np.random.randint(1, 5) * 60
            for i in range(num_txns):
                txn_timestamp = date_obj + timedelta(
                    seconds=start_time + i * np.random.randint(1, interval_seconds)
                )
                txn_date = txn_timestamp.strftime(self._date_format)
                txn_timestamp = txn_timestamp.strftime(self._timestamp_format)
                if i == 0:
                    txn_value = round(random.uniform(5, 10), 2)
                else:
                    txn_value = round(
                        txn_value_increment, 2
                    )  # use a fixed rounded number (for clearer difference from scenario 2)
                fraudulent_txns.append(
                    {
                        "customer_id": customer_id,
                        "txn_id": self._uuid_generator.generate_id(prefix="t_"),
                        "txn_timestamp": txn_timestamp,
                        "txn_date": txn_date,
                        "txn_value": txn_value,
                        "txn_fraud": 1,
                        "txn_fraud_scenario": 3,
                    }
                )
        else:
            raise ValueError("Invalid scenario number. Choose 1, 2 or 3.")
        return fraudulent_txns


In [122]:
# Payment Channel Features
# out of all the frauds, this is the percentage of frauds that are of each type
payment_channel_fraud = {"cnp": 0.75, "digital_wallet": 0.15, "pos": 0.1}

# Customer Features
# if the customer enables 2FA, the probability of fraud is 0.5% otherwise it is 5%
two_fa_fraud = {1: 0.005, 0: 0.05}

# groups that are more susceptible to frauds are millenials and seniors
age_group_fraud = {
    "18-25": 0.05,
    "26-35": 0.05,
    "36-45": 0.01,
    "46-55": 0.01,
    "56-65": 0.05,
    "66+": 0.05,
}

# groups with increasingly higher spending are more susceptible to frauds:
spending_group_fraud = {
    "low": 0.005,
    "low-middle": 0.0075,
    "middle": 0.01,
    "high-middle": 0.0125,
    "high": 0.015,
}

age_config = {
    "name": ["18-25", "26-35", "36-45", "46-55", "56-65", "66+"],
    "customer_pct": [0.1, 0.15, 0.2, 0.25, 0.15, 0.15],
    "fraud_proba": [0.05, 0.05, 0.01, 0.01, 0.05, 0.05],
}

spending_config = {
    "name": ["low", "low-middle", "middle", "high-middle", "high"],
    "customer_pct": [0.1, 0.25, 0.35, 0.25, 0.05],
    "fraud_proba": [0.005, 0.0075, 0.01, 0.0125, 0.015],
}

two_fa_config = {
    "name": ["no", "yes"],
    "customer_pct": [0.5, 0.5],
    "fraud_proba": [0.05, 0.005],
}


class CustomerProfileGenerator:
    def __init__(
        self,
        age_config: dict,
        spending_config: dict,
        two_fa_config: dict,
    ):
        self.age_config = age_config
        self.spending_config = spending_config
        self.two_fa_config = two_fa_config

        self._fraud_proba_weights = {
            "two_fa": 0.5,
            "spending_group": 0.3,
            "age_group": 0.2,
        }

    def generate_profile_with_fraud_exposure(self):
        age_group = np.random.choice(
            self.age_config["name"], p=self.age_config["customer_pct"]
        ).item()

        spending_group = np.random.choice(
            self.spending_config["name"], p=self.spending_config["customer_pct"]
        ).item()

        two_fa = np.random.choice(
            self.two_fa_config["name"], p=self.two_fa_config["customer_pct"]
        ).item()

        age_group_fraud_proba = self.age_config["fraud_proba"][
            self.age_config["name"].index(age_group)
        ]

        spending_group_fraud_proba = self.spending_config["fraud_proba"][
            self.spending_config["name"].index(spending_group)
        ]

        two_fa_fraud_proba = self.two_fa_config["fraud_proba"][
            self.two_fa_config["name"].index(two_fa)
        ]

        profile = {
            "age_group": [age_group, age_group_fraud_proba],
            "spending_group": [spending_group, spending_group_fraud_proba],
            "two_fa": [two_fa, two_fa_fraud_proba],
        }
        return profile


customer_profile_generator = CustomerProfileGenerator(
    age_config, spending_config, two_fa_config
)
customer_profile_generator.generate_profile_with_fraud_exposure()

{'age_group': ['46-55', 0.01],
 'spending_group': ['middle', 0.01],
 'two_fa': ['yes', 0.005]}

In [126]:
# generate 1000 custemer profiles and check the distribution 
profiles = [customer_profile_generator.generate_profile_with_fraud_exposure() for i in range(1000)]
age_groups = [profile["age_group"][0] for profile in profiles]
spending_groups = [profile["spending_group"][0] for profile in profiles]
two_fa = [profile["two_fa"][0] for profile in profiles]

# calculate the distribution of these features
age_group_dist = {age: age_groups.count(age) / 1000 for age in set(age_groups)}
spending_group_dist = {
    spending: spending_groups.count(spending) / 1000 for spending in set(spending_groups)
}
two_fa_dist = {two_fa: two_fa.count(two_fa) / 1000 for two_fa in set(two_fa)}

print(age_group_dist)
print(spending_group_dist)
print(two_fa_dist)

{'46-55': 0.267, '18-25': 0.091, '66+': 0.147, '36-45': 0.194, '26-35': 0.153, '56-65': 0.148}
{'low-middle': 0.236, 'high-middle': 0.251, 'low': 0.099, 'high': 0.045, 'middle': 0.369}
{'no': 0.001, 'yes': 0.001}


In [ ]:
spending_habits = {
    'name': ['low', 'low-middle', 'middle', 'high-middle', 'high'],
    'txn_mean_low': [5, 20, 40, 60, 80],
    'txn_mean_high': [20, 40, 60, 80, 100],
    'txn_cv_low': [0.3, 0.4, 0.5, 0.6, 0.7],
    'txn_cv_high': [0.4, 0.5, 0.6, 0.7, 0.8],
    'txn_lambda': [0.25, 0.5, 1, 1.5, 2] # to simplify, we assume middle income person has average 1 txn per day
}

customer_profile_generator = CustomerProfileGenerator(
    age_config, spending_config, two_fa_config
)
customer_generator = CustomerGenerator(spending_habits)

customer_profile = customer_profile_generator.generate_profile_with_fraud_exposure()
print(customer_profile) # without spending habit
customer = customer_generator.generate_customer_from_profile(customer_profile)
customer.profile # with spending habit

{'age_group': ['36-45', 0.01], 'spending_group': ['low', 0.005], 'two_fa': ['no', 0.05]}


{'age_group': ['36-45', 0.01],
 'spending_group': ['low', 0.005],
 'two_fa': ['no', 0.05],
 'txn_mean': 6.36,
 'txn_mu': 1.8041491647322603,
 'txn_sigma': 0.3029165318029716,
 'txn_lambda': 0.25}

In [ ]:
# as these features are usually not independet from each other, e.g., it is reasonable that a senior is more likely to have a high spending, while less tech-savy to enable 2FA, we should not use a multiplication of the probabilities to calculate the final probability of fraud. Instead, we can use a weighted average model:
def weighted_average(two_factor_auth, age_group, spending_group):
    weights = {
        "two_factor_auth": 0.5,
        "spending_group": 0.3,
        "age_group": 0.2,
    }

    probability = (
        weights["two_factor_auth"] * two_factor_auth_fraud[two_factor_auth]
        + weights["age_group"] * age_group_fraud[age_group]
        + weights["spending_group"] * spending_group_fraud[spending_group]
    )
    return probability


sample_probability = weighted_average(1, "66+", "high")
sample_probability

# calculate the probability of all the combinations of the features:
fraud_probabilities = {}
for two_factor_auth in two_factor_auth_fraud:
    for age_group in age_group_fraud:
        for spending_group in spending_group_fraud:
            probability = round(
                weighted_average(two_factor_auth, age_group, spending_group), 6
            )
            fraud_probabilities[(two_factor_auth, age_group, spending_group)] = (
                probability
            )
fraud_probabilities

# for simplicity, we can ignore the relationship between the customer feature such as age and spending groups, or age and 2FA, and split the total number of customers equally among the combinations of the features:
total_customers = 100000

# create a list of all the combinations of the features
# this is equivalent to nested for loops
combinations = list(product(two_factor_auth_fraud, age_group_fraud, spending_group_fraud))

customers_per_combination = total_customers / len(combinations)
customers_per_combination

1666.6666666666667